In [13]:
# Import necessary libraries
import numpy as np
import pandas as pd
import os
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Conv2D
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [14]:
# === DATA LOADING & PREPROCESSING ===

# Set dataset paths (Update if needed)
base_dir = "/Users/mac/Development/diploma/skin-mutation-detection/ham10000"
image_dir = os.path.join(base_dir, "HAM10000_images_part_1")  # Adjust if images are in two folders
metadata_path = os.path.join(base_dir, "HAM10000_metadata.csv")

In [15]:
# Load metadata
df = pd.read_csv(metadata_path)

# Define class labels
lesion_type_dict = {
    'nv': 'Melanocytic nevi',
    'mel': 'Melanoma',
    'bkl': 'Benign keratosis-like lesions',
    'bcc': 'Basal cell carcinoma',
    'akiec': 'Actinic keratoses',
    'vasc': 'Vascular lesions',
    'df': 'Dermatofibroma'
}

# Map labels to integers
df['label'] = df['dx'].map({'nv': 0, 'mel': 1, 'bkl': 2, 'bcc': 3, 'akiec': 4, 'vasc': 5, 'df': 6})


In [16]:

# Load images
IMG_SIZE = 224  # EfficientNetB3 requires 224x224
def load_and_preprocess_image(image_id):
    image_path = os.path.join(image_dir, image_id + ".jpg")
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))  # Resize
    image = image / 255.0  # Normalize
    return image

# Load all images
X = np.array([load_and_preprocess_image(img_id) for img_id in df['image_id']])
y = np.array(df['label'])

In [17]:
# Convert labels to one-hot encoding
y = to_categorical(y, num_classes=7)

# Split dataset
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Print dataset shapes
print(f"Training Set: {X_train.shape}, {y_train.shape}")
print(f"Validation Set: {X_val.shape}, {y_val.shape}")


Training Set: (8012, 224, 224, 3), (8012, 7)
Validation Set: (2003, 224, 224, 3), (2003, 7)


In [18]:
# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

In [19]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, Multiply, Activation, Concatenate, Reshape
from tensorflow.keras import backend as K

class SoftAttention(Layer):
    def __init__(self, ch, m, concat_with_x=False, aggregate=False, **kwargs):
        super(SoftAttention, self).__init__(**kwargs)
        self.channels = int(ch)
        self.multiheads = m
        self.aggregate_channels = aggregate
        self.concat_input_with_scaled = concat_with_x

    def build(self, input_shape):
        self.i_shape = input_shape  # (batch, height, width, channels)
        
        # Define Conv2D for EfficientNet compatibility
        self.conv2d = Conv2D(filters=self.multiheads, kernel_size=3, padding='same', activation='relu', kernel_initializer='he_uniform')
        
        super(SoftAttention, self).build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]
        height, width, channels = self.i_shape[1:]

        # Apply 2D convolution
        conv2d = self.conv2d(x)  # (batch, height, width, multiheads)

        # Compute softmax attention weights
        softmax_alpha = tf.nn.softmax(conv2d, axis=-1)
        softmax_alpha = Reshape(target_shape=(height, width, self.multiheads, 1))(softmax_alpha)

        # Expand input feature map to match attention weights
        x_exp = tf.expand_dims(x, axis=-2)  # (batch, height, width, 1, channels)
        
        # Element-wise multiplication (apply attention weights)
        u = Multiply()([softmax_alpha, x_exp])  

        # Reshape to match expected output
        u = tf.reshape(u, shape=[batch_size, height, width, channels * self.multiheads])

        # Concatenation (optional)
        if self.concat_input_with_scaled:
            o = Concatenate(axis=-1)([u, x])
        else:
            o = u

        return [o, softmax_alpha]

    def compute_output_shape(self, input_shape):
        return [input_shape[:-1] + (input_shape[-1] * self.multiheads,), input_shape]


In [20]:
# === EfficientNet + SoftAttention Model ===

# Load Pretrained EfficientNet
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Freeze base model layers (for transfer learning)
base_model.trainable = False

# Add SoftAttention after feature extraction
x = base_model.output
x = Conv2D(256, (3, 3), activation="relu", padding="same")(x)
x, attention_maps = SoftAttention(ch=256, m=8)(x)  # Using 8 attention heads
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)  # Regularization
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(7, activation="softmax")(x)

# Create final model
model = Model(inputs=base_model.input, outputs=output)

In [ ]:

# Compile the model
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

# === TRAINING ===
history = model.fit(datagen.flow(X_train, y_train, batch_size=32),
                    validation_data=(X_val, y_val),
                    epochs=20,
                    verbose=1)

# Save the trained model
model.save("soft_attention_skin_model.h5")

/Users/mac/Library/Caches/pypoetry/virtualenvs/diploma-JhBIondL-py3.11/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 111s 358ms/step - accuracy: 0.6499 - loss: 1.2622 - val_accuracy: 0.6695 - val_loss: 1.1360
Epoch 2/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 69s 274ms/step - accuracy: 0.6582 - loss: 1.2019 - val_accuracy: 0.6695 - val_loss: 1.1357
Epoch 3/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 68s 272ms/step - accuracy: 0.6744 - loss: 1.1756 - val_accuracy: 0.6695 - val_loss: 1.1355
Epoch 4/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 68s 270ms/step - accuracy: 0.6698 - loss: 1.1769 - val_accuracy: 0.6695 - val_loss: 1.1305
Epoch 5/20
 44/251 ━━━━━━━━━━━━━━━━━━━━ 45s 221ms/step - accuracy: 0.6812 - loss: 1.1417

In [ ]:
# === RESULTS ===
plt.plot(history.history["accuracy"], label="train_accuracy")
plt.plot(history.history["val_accuracy"], label="val_accuracy")
plt.legend()
plt.show()